# Week 10 — BBO capstone driver

Round 10. Per-function response to W9, with step sizes tuned by what each result implied.

- **F2, F7** — continuations paid last round, so continue at a reduced step.
- **F3, F4, F6** — mirrors failed, so step *away* from the mirror direction.
- **F5** — the small W8→W9 shift was the wrong way; reverse it at half scale.
- **F8** — W8 and W9 bracket a maximum, so submit the midpoint. Three collinear points then define a parabola, which is the first use of the technique that carries rounds 11–13.
- **F1** — a wide exploratory span, 0.15 to 0.85.

**Note on F4:** the proposed x3 lands on 0.001, the domain floor. Clamped rather than rejected, which makes that coordinate uninformative.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 10
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 10
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: 'wide exploratory span',
    2: 'continue at reduced step',
    3: 'step away from failed mirror',
    4: 'step away from failed mirror (x3 clamped)',
    5: 'reverse failed step at 50%',
    6: 'step away from failed mirror',
    7: 'continue at 20% step',
    8: 'midpoint of W8/W9 bracket',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 9. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — response tuned per function

The F8 midpoint is the interesting one: three collinear points are enough to fit a parabola and solve for its vertex.

In [ ]:
proposals = {
    1: np.array([0.15, 0.85]),
    2: np.array([0.754432, 0.849899]),
    3: np.array([0.161275, 0.724316, 0.566951]),
    4: np.array([0.152272, 0.021888, 0.001, 0.944762]),
    5: np.array([0.301806, 0.737874, 0.084498, 0.668917]),
    6: np.array([0.052822, 0.531961, 0.970967, 0.838837, 0.217949]),
    7: np.array([0.630359, 0.403096, 0.524833, 0.586005, 0.38784, 0.556038]),
    8: np.array([0.038983, 0.275485, 0.181927, 0.347803, 0.803194, 0.234763, 0.936041, 0.094563]),
}

# F8: confirm the submitted point is the midpoint of the W8/W9 pair.
mid = (np.array(bbo.HISTORY[8][8][0]) + np.array(bbo.HISTORY[9][8][0])) / 2
print('F8 midpoint check:', np.allclose(mid, proposals[8], atol=1e-6))

# Any coordinate resting on the domain floor is effectively unmeasured.
pd.DataFrame([dict(func=f"F{fid}",
                   at_lower_bound=int((proposals[fid] <= bbo.LOW + 1e-9).sum()),
                   submission=bbo.submission(proposals[fid]))
              for fid in bbo.FUNC_IDS])


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.669937, 0.751452],
    2: [0.354432, 0.449899],
    3: [0.111275, 0.774316, 0.516951],
    4: [0.183872, 0.065353, 0.017618, 0.904326],
    5: [0.308806, 0.730874, 0.091998, 0.661917],
    6: [0.070021, 0.530732, 0.952853, 0.825805, 0.228797],
    7: [0.965568, 0.153915, 0.588691, 0.807159, 0.099427, 0.700138],
    8: [0.038733, 0.275735, 0.181777, 0.348003, 0.803094, 0.234913, 0.935841, 0.094663],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 10 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: 2.8557e-181,
#     2: 0.402932,
#     3: -0.047062,
#     4: -27.222858,
#     5: 1.326997,
#     6: -1.014444,
#     7: 0.49933,
#     8: 8.750177,
# }
#
# F5 1.327 and F7 0.499 improve. F8's three collinear values are near-perfectly
# linear and almost flat, which says the micro-nudge ray is exhausted - the 8.75
# plateau is a false summit, though it takes another round to prove it.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
